In [1]:
import sys
from pathlib import Path

package_root = str(Path.cwd().parent)
if package_root not in sys.path:
    sys.path.append(package_root)

In [2]:
from __future__ import annotations
import pandas as pd 
from typing import Dict, List, Tuple, Set
from pathlib import Path
from collections.abc import Iterable
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import time
import re
import numpy as np
from copy import deepcopy
from datetime import datetime
from typing import Any
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [3]:
pd.set_option("display.max_colwidth", None)

In [4]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"

In [5]:
MAPPING_LIST_1 = f"{DEVICE_MAPPING_FOLDER}/{DATASET1}/list_of_devices_processed.csv"
MAPPING_LIST_2 = f"{DEVICE_MAPPING_FOLDER}/{DATASET2}/list_of_devices_processed.csv"
INPUT_1 = f"{INPUT_FOLDER}/{DATASET1}"
INPUT_2 = f"{INPUT_FOLDER}/{DATASET2}"

In [6]:
csv_files_1 = [f"{INPUT_1}/{f.name}" for f in sorted(Path(INPUT_1).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in csv_files_1:
    print(file)

../../../dataset/processed-data/input-data/dataset1/16-09-23.csv
../../../dataset/processed-data/input-data/dataset1/16-09-24.csv
../../../dataset/processed-data/input-data/dataset1/16-09-25.csv
../../../dataset/processed-data/input-data/dataset1/16-09-26.csv
../../../dataset/processed-data/input-data/dataset1/16-09-27.csv
../../../dataset/processed-data/input-data/dataset1/16-09-28.csv
../../../dataset/processed-data/input-data/dataset1/16-09-29.csv
../../../dataset/processed-data/input-data/dataset1/16-09-30.csv
../../../dataset/processed-data/input-data/dataset1/16-10-01.csv
../../../dataset/processed-data/input-data/dataset1/16-10-02.csv
../../../dataset/processed-data/input-data/dataset1/16-10-03.csv
../../../dataset/processed-data/input-data/dataset1/16-10-04.csv
../../../dataset/processed-data/input-data/dataset1/16-10-05.csv
../../../dataset/processed-data/input-data/dataset1/16-10-06.csv
../../../dataset/processed-data/input-data/dataset1/16-10-07.csv
../../../dataset/processe

In [7]:
device_mapping_1 = pd.read_csv(MAPPING_LIST_1)
device_dict_1 = (
    device_mapping_1
    .set_index("mac_address")
    .drop(columns=["mac_address"], errors="ignore")
    .to_dict(orient="index")
)
device_dict_1

{'d0:52:a8:00:67:5e': {'device_name': 'Smart Things',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'hub',
  'vendor': 'Samsung SmartThings'},
 '44:65:0d:56:cc:d3': {'device_name': 'Amazon Echo',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'smart speaker',
  'vendor': 'Amazon'},
 '70:ee:50:18:34:43': {'device_name': 'Netatmo Welcome',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Netatmo'},
 'f4:f2:6d:93:51:f1': {'device_name': 'TP-Link Day Night Cloud camera',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'TP-Link'},
 '00:16:6c:ab:6b:88': {'device_name': 'Samsung SmartCam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Samsung'},
 '30:8c:fb:2f:e4:b2': {'device_name': 'Dropcam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Dropcam'},
 '00:62:6e:51:27:2e': {'device_name': 'Insteon Camera Wired',
  'ip_address': 

In [ ]:
RANDOM_STATE = 42
WINDOW_SECONDS = 60

In [9]:
# Outgoing volume
def extract_data_from_pcap(
    file_paths: Iterable[str]
) -> pd.DataFrame:
    columns = [
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
    ]

    all_data = []

    for file_path in file_paths:
        raw_df = pd.read_csv(
            file_path,
            usecols=columns,
            dtype={
                "frame.time_epoch": "float64",
                "frame.len": "int32",
                "eth.src": "category",
                "eth.dst": "category",
            }
        )

        raw_df["capture_id"] = Path(file_path).stem
        all_data.append(raw_df)


    res = pd.concat(all_data, ignore_index=True)
    res = res.sort_values(
        ["frame.time_epoch", "eth.src"],
        ignore_index=True,
    )

    return res


In [10]:
all_data = extract_data_from_pcap(csv_files_1)

In [11]:
all_data.head(10)

,frame.time_epoch,frame.len,eth.src,eth.dst,capture_id
0,1.474553e+09,70,18:b7:9e:02:20:44,14:cc:20:51:33:ea,16-09-23
1,1.474553e+09,66,18:b7:9e:02:20:44,14:cc:20:51:33:ea,16-09-23
2,1.474553e+09,66,14:cc:20:51:33:ea,30:8c:fb:2f:e4:b2,16-09-23
3,1.474553e+09,60,d0:52:a8:00:67:5e,14:cc:20:51:33:ea,16-09-23
4,1.474553e+09,98,14:cc:20:51:33:ea,70:ee:50:18:34:43,16-09-23
5,1.474553e+09,88,70:ee:50:18:34:43,14:cc:20:51:33:ea,16-09-23
6,1.474553e+09,68,14:cc:20:51:33:ea,18:b7:9e:02:20:44,16-09-23
7,1.474553e+09,54,14:cc:20:51:33:ea,d0:52:a8:00:67:5e,16-09-23
8,1.474553e+09,42,ec:1a:59:79:f4:89,14:cc:20:51:33:ea,16-09-23
9,1.474553e+09,42,14:cc:20:51:33:ea,ec:1a:59:79:f4:89,16-09-23


In [12]:
eda_df = all_data.copy()

In [13]:
capture_counts = (
    eda_df.groupby("eth.src")["capture_id"]
      .nunique()
)

single_capture_devices = capture_counts[
    capture_counts == 1
].index

single_capture_info = (
    eda_df[eda_df["eth.src"].isin(single_capture_devices)]
    [
        ["eth.src"]
    ]
    .drop_duplicates()
    .sort_values("eth.src")
)
# single_capture_info = label_features(single_capture_info, device_dict_1)
# single_capture_info_with_label = label_features(single_capture_info, device_dict_1)
# single_capture_info
eth_src_list = single_capture_info["eth.src"].dropna().unique().tolist()
print(f"Number of devices seen in only one capture: {len(eth_src_list)}")
for mac in eth_src_list:
    if mac in device_dict_1:
        print(f"mac '{mac}': {device_dict_1[mac]['device_name']}")
    else:
        print(f"mac '{mac}': not found in device list")

Number of devices seen in only one capture: 6
mac '00:24:e4:10:ee:4c': not found in device list
mac '30:8c:fb:b6:ea:45': Nest Dropcam
mac '8a:05:81:fa:cc:14': not found in device list
mac 'd2:13:91:23:2a:58': not found in device list
mac 'e8:ab:fa:19:de:4f': Insteon Camera Wireless
mac 'f4:5c:89:93:cc:85': MacBook/Iphone


In [14]:
device_to_be_include = set(device_dict_1) - set(eth_src_list)
len(device_to_be_include)

28

In [15]:
list_of_devices_to_be_classified = list(device_to_be_include)
print(f"Total: {len(list_of_devices_to_be_classified)} devices")
for mac in list_of_devices_to_be_classified:
    print(f"mac '{mac}': {device_dict_1[mac]["device_name"]},\tis_iot: {device_dict_1[mac]["is_iot_device"]}, \tdevice_type: {device_dict_1[mac]["device_type"]}")

Total: 28 devices
mac '00:16:6c:ab:6b:88': Samsung SmartCam,	is_iot: 1, 	device_type: camera
mac '74:c6:3b:29:d7:1d': iHome,	is_iot: 1, 	device_type: smart plug
mac '18:b7:9e:02:20:44': Triby Speaker,	is_iot: 1, 	device_type: smart speaker
mac 'd0:52:a8:00:67:5e': Smart Things,	is_iot: 1, 	device_type: hub
mac '00:24:e4:1b:6f:96': Withings Smart scale,	is_iot: 1, 	device_type: smart scale
mac '18:b4:30:25:be:e4': NEST Protect smoke alarm,	is_iot: 1, 	device_type: smoke alarm
mac '50:c7:bf:00:56:39': TP-Link Smart plug,	is_iot: 1, 	device_type: smart plug
mac '30:8c:fb:2f:e4:b2': Dropcam,	is_iot: 1, 	device_type: camera
mac 'ec:1a:59:83:28:11': Belkin wemo motion sensor,	is_iot: 1, 	device_type: motion sensor
mac 'd0:a6:37:df:a1:e1': IPhone,	is_iot: 0, 	device_type: nan
mac '14:cc:20:51:33:ea': TPLink Router Bridge LAN (Gateway),	is_iot: 0, 	device_type: nan
mac 'f4:f2:6d:93:51:f1': TP-Link Day Night Cloud camera,	is_iot: 1, 	device_type: camera
mac '70:ee:50:18:34:43': Netatmo Welcome,

In [16]:
process_df = all_data.copy()

In [ ]:
def get_all_devices(list_of_devices: List[str]) -> Set[str]:
    return set(list_of_devices)
def drop_router(list_of_devices: List[str]) -> Set[str]:
    router_devices = set(["14:cc:20:51:33:ea"])
    return set(list_of_devices) - router_devices
def drop_router_and_hub(list_of_devices: List[str]) -> Set[str]:
    router_devices = set(["14:cc:20:51:33:ea"])
    hub_devices = set(["d0:52:a8:00:67:5e","44:65:0d:56:cc:d3"])
    return set(list_of_devices) - router_devices - hub_devices
def drop_router_and_camera(list_of_devices: List[str]) -> Set[str]:
    router_devices = set(["14:cc:20:51:33:ea"])
    camera_devices = set(["70:ee:50:18:34:43","30:8c:fb:2f:e4:b2","00:16:6c:ab:6b:88","f4:f2:6d:93:51:f1","00:62:6e:51:27:2e"])
    return set(list_of_devices) - router_devices - camera_devices

dropping_strategies = {
    "no_dropping": get_all_devices,
    "drop_router": drop_router,
    "drop_router_and_hub": drop_router_and_hub,
    "drop_router_and_camera": drop_router_and_camera,
}

In [4]:
def label_2_class(features: pd.DataFrame, device_mapping: Dict[str, Dict[str,str]]) -> pd.DataFrame:
    unknown_macs = set(features["mac"]) - set(device_mapping)
    if unknown_macs:
        raise KeyError(
            f"MAC addresses not found in device_mapping: {unknown_macs}"
        )
    res = features.copy()
    res["is_iot_device"] = res["mac"].map(
        lambda x: device_mapping[x]["is_iot_device"]
    )
    res["device_type"] = res["mac"].map(
        lambda x: device_mapping[x]["device_type"]
    )
    res["vendor"] = res["mac"].map(
        lambda x: device_mapping[x]["vendor"]
    )

    return res
def label_3_class_health_care_device_type_3(features: pd.DataFrame, device_mapping: Dict[str, Dict[str,str]]) -> pd.DataFrame:
    unknown_macs = set(features["mac"]) - set(device_mapping)
    if unknown_macs:
        raise KeyError(
            f"MAC addresses not found in device_mapping: {unknown_macs}"
        )
    device_mapping_copy = deepcopy(device_mapping)
    health_monitor_devices = set(["74:6a:89:00:2e:25","18:b4:30:25:be:e4","00:24:e4:1b:6f:96"])
    for mac in device_mapping_copy:
        if device_mapping_copy[mac]["is_iot_device"] == 0:
            device_mapping_copy[mac]["is_iot_device"] = 1
        else:
            if mac in health_monitor_devices:
                device_mapping_copy[mac]["is_iot_device"] = 3
            else:
                device_mapping_copy[mac]["is_iot_device"] = 2
    res = features.copy()
    res["is_iot_device"] = res["mac"].map(
        lambda x: device_mapping[x]["is_iot_device"]
    )
    res["device_type"] = res["mac"].map(
        lambda x: device_mapping[x]["device_type"]
    )
    res["vendor"] = res["mac"].map(
        lambda x: device_mapping[x]["vendor"]
    )

    return res
def label_3_class_camera_device_type_3(features: pd.DataFrame, device_mapping: Dict[str, Dict[str,str]]) -> pd.DataFrame:
    unknown_macs = set(features["mac"]) - set(device_mapping)
    if unknown_macs:
        raise KeyError(
            f"MAC addresses not found in device_mapping: {unknown_macs}"
        )
    device_mapping_copy = deepcopy(device_mapping)
    camera_set = set(["30:8c:fb:b6:ea:45", "00:62:6e:51:27:2e", "e8:ab:fa:19:de:4f", "30:8c:fb:2f:e4:b2", "00:16:6c:ab:6b:88","f4:f2:6d:93:51:f1","70:ee:50:18:34:43"])
    for mac in device_mapping_copy:
        if device_mapping_copy[mac]["is_iot_device"] == 0:
            device_mapping_copy[mac]["is_iot_device"] = 1
        else:
            if mac in camera_set:
                device_mapping_copy[mac]["is_iot_device"] = 3
            else:
                device_mapping_copy[mac]["is_iot_device"] = 2
    res = features.copy()
    res["is_iot_device"] = res["mac"].map(
        lambda x: device_mapping[x]["is_iot_device"]
    )
    res["device_type"] = res["mac"].map(
        lambda x: device_mapping[x]["device_type"]
    )
    res["vendor"] = res["mac"].map(
        lambda x: device_mapping[x]["vendor"]
    )

    return res

labeling_strategies = {
    "2_class": label_2_class,
    "3_class_health_care_device_type_3": label_3_class_health_care_device_type_3,
    "3_class_camera_device_type_3": label_3_class_camera_device_type_3,
}   

In [ ]:
def build_feature(
    raw_df: pd.DataFrame,
    device_to_be_include: set,
    window: int = WINDOW_SECONDS
) -> pd.DataFrame:

    if window <= 0:
        raise ValueError("window must be greater than 0")

    all_features = []

    df = raw_df.copy()

    for capture_id, capture_df in df.groupby(
        "capture_id",
        sort=False,
        observed=True,
    ):
        capture_df = capture_df.copy()

        devices = capture_df["eth.src"].dropna().unique()

        devices = np.array([
            device
            for device in devices
            if device in device_to_be_include
        ])

        if len(devices) == 0:
            continue

        capture_start = capture_df["frame.time_epoch"].min()

        capture_df["window"] = (
            (capture_df["frame.time_epoch"] - capture_start) // window
        ).astype("int32")

        features = (
            capture_df[
                capture_df["eth.src"].isin(devices)
            ]
            .groupby(
                ["window", "eth.src"],
                sort=False,
                observed=True,
            )
            .agg(
                packet_count=("frame.len", "size"),
                total_bytes=("frame.len", "sum"),
                first_timestamp=("frame.time_epoch", "min"),
                last_timestamp=("frame.time_epoch", "max"),
            )
            .reset_index()
        )

        features = features.sort_values(
            ["eth.src", "window"]
        )

        features["previous_last_timestamp"] = (
            features
            .groupby(
                "eth.src",
                observed=True
            )["last_timestamp"]
            .shift(1)
        )

        features["idle_time"] = np.where(
            features["previous_last_timestamp"].isna(),
            features["first_timestamp"] - capture_start,
            features["first_timestamp"] - features["previous_last_timestamp"]
        )

        features["capture_id"] = capture_id

        features = features[
            [
                "capture_id",
                "window",
                "eth.src",
                "packet_count",
                "total_bytes",
                "idle_time",
            ]
        ]

        all_features.append(features)

    if not all_features:
        return pd.DataFrame(
            columns=[
                "capture_id",
                "window",
                "eth.src",
                "packet_count",
                "total_bytes",
                "idle_time",
            ]
        )

    result = pd.concat(
        all_features,
        ignore_index=True,
    )

    result["packet_count"] = result["packet_count"].astype("int64")
    result["total_bytes"] = result["total_bytes"].astype("int64")
    result["idle_time"] = result["idle_time"].astype("float64")

    return result[
        [
            "capture_id",
            "window",
            "eth.src",
            "packet_count",
            "total_bytes",
            "idle_time",
        ]
    ]